# Pearls AQI Predictor — Exploratory Data Analysis

Quick-start EDA notebook. Run the synthetic data generator first if you don't have
real data yet:

```bash
python scripts/generate_synthetic_data.py
```

or run the real backfill:

```bash
python -m src.data.backfill_historical
```

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.express as px

from src import config
from src.features.feature_store import get_feature_store

store = get_feature_store()
df = store.read_features()
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df.shape

In [ ]:
# AQI distribution per city
px.box(df, x='city', y='aqi', title='AQI distribution by city')

In [ ]:
# Hourly AQI trend for one city
city_df = df[df['city'] == 'Lahore'].sort_values('timestamp')
px.line(city_df, x='timestamp', y='aqi', title='Lahore — AQI over time')

In [ ]:
# Average AQI by hour of day (diurnal pattern — traffic/industry cycles)
hourly = df.groupby(['city', 'hour'])['aqi'].mean().reset_index()
px.line(hourly, x='hour', y='aqi', color='city', title='Average AQI by hour of day')

In [ ]:
# Correlation between weather features and AQI
corr_cols = ['aqi', 'temp', 'humidity', 'pressure', 'wind_speed', 'clouds'] + config.POLLUTANTS
corr = df[corr_cols].corr()
px.imshow(corr, text_auto='.2f', title='Feature correlation with AQI')

In [ ]:
# Pollutant concentration trends
melted = df[df['city'] == 'Lahore'].melt(
    id_vars=['timestamp'], value_vars=config.POLLUTANTS, var_name='pollutant', value_name='concentration'
)
px.line(melted, x='timestamp', y='concentration', color='pollutant', title='Lahore — pollutant concentrations (µg/m³)')